In [1]:
# ── 0. Installs ──────────────────────────────────────────────
import subprocess, sys, gc, os, json, pickle, random, shutil, yaml
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

try:
    import ultralytics
except ImportError:
    pip_install("ultralytics")

import torch
from ultralytics import YOLO
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_curve, auc, confusion_matrix
)
from collections import defaultdict

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device  :", DEVICE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch : 2.10.0+cu128
CUDA    : True
Device  : cuda


In [2]:
# ── 1. Paths & Hyper-parameters ──────────────────────────────
SRC_BASE  = "/kaggle/input/datasets/kavyagupta3011/vr-dataset-deepfashion2/pruned_deepfashion2/final_dataset"
SPLIT_PKL = "/kaggle/input/datasets/kavyagupta3011/vr-dataset-deepfashion2/split.pkl"

SRC_TRAIN_IMG = os.path.join(SRC_BASE, "train", "images")
SRC_TRAIN_ANN = os.path.join(SRC_BASE, "train", "annotations")
SRC_VAL_IMG   = os.path.join(SRC_BASE, "val",   "images")
SRC_VAL_ANN   = os.path.join(SRC_BASE, "val",   "annotations")

OUTPUT_DIR    = "/kaggle/working"
YOLO_DIR      = os.path.join(OUTPUT_DIR, "yolo_dataset")
RESULTS_DIR   = os.path.join(OUTPUT_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Memory-safe hyper-params ─────────────────────────────────
IMG_SIZE          = 320       # lower → less VRAM
BATCH_SIZE        = 8         # safe for 16 GB GPU
NUM_EPOCHS        = 5        # enough for demo; bump to 10+ for real training
LR_TRANSFER       = 5e-4
LR_SCRATCH        = 1e-3
SUBSAMPLE_STEP    = 3         # use every 5th image to stay within memory
IOU_THRESHOLD     = 0.5
CONF_THRESHOLD    = 0.25

# ── Label mappings ───────────────────────────────────────────
top5 = {1, 2, 7, 8, 9}

CAT_TO_IDX = {1: 0, 8: 1, 7: 2, 2: 3, 9: 4}
IDX_TO_NAME = {
    0: "short_sleeve_top",
    1: "trousers",
    2: "shorts",
    3: "long_sleeve_top",
    4: "skirt",
}
NUM_CLASSES = 5

with open(os.path.join(OUTPUT_DIR, "label_map.json"), "w") as f:
    json.dump({v: k for k, v in IDX_TO_NAME.items()}, f, indent=4)
print("Saved label_map.json")

Saved label_map.json


In [3]:
# ── 2. Train / Val Split ─────────────────────────────────────
def build_split(val_size=0.20, seed=42):
    valid_files, strat_labels = [], []
    for ann_file in sorted(os.listdir(SRC_TRAIN_ANN)):
        if not ann_file.endswith(".json"):
            continue
        img_file = ann_file.replace(".json", ".jpg")
        if not os.path.exists(os.path.join(SRC_TRAIN_IMG, img_file)):
            continue
        with open(os.path.join(SRC_TRAIN_ANN, ann_file)) as fh:
            data = json.load(fh)
        cats = [v["category_id"] for v in data.values()
                if isinstance(v, dict) and v.get("category_id") in top5]
        if not cats:
            continue
        valid_files.append(img_file)
        strat_labels.append(max(set(cats), key=cats.count))
    tr, vl = train_test_split(valid_files, test_size=val_size,
                               stratify=strat_labels, random_state=seed)
    return tr, vl

if os.path.exists(SPLIT_PKL):
    with open(SPLIT_PKL, "rb") as fh:
        saved = pickle.load(fh)
    train_files_full = saved["train_files"]
    val_files_full   = saved["val_files"]
else:
    train_files_full, val_files_full = build_split()
    with open(SPLIT_PKL, "wb") as fh:
        pickle.dump({"train_files": train_files_full,
                     "val_files": val_files_full}, fh)

train_files = train_files_full[::SUBSAMPLE_STEP]
val_files   = val_files_full[::SUBSAMPLE_STEP]
print(f"Train images : {len(train_files)}")
print(f"Val   images : {len(val_files)}")

Train images : 38447
Val   images : 9612


In [4]:
# ── 3. Build YOLO Dataset ────────────────────────────────────
def poly_to_bbox_norm(poly_x, poly_y, W, H):
    """Return (cx, cy, bw, bh) normalised to [0,1]."""
    x1, y1, x2, y2 = min(poly_x), min(poly_y), max(poly_x), max(poly_y)
    cx = ((x1 + x2) / 2) / W
    cy = ((y1 + y2) / 2) / H
    bw = (x2 - x1) / W
    bh = (y2 - y1) / H
    return cx, cy, bw, bh

def polygon_to_yolo_seg(poly_x, poly_y, W, H):
    """Interleave normalised x,y pairs for YOLO seg format."""
    pts = []
    for px, py in zip(poly_x, poly_y):
        pts += [px / W, py / H]
    return pts

def write_split(file_list, src_img_dir, src_ann_dir, dst_img_dir, dst_lbl_dir):
    """Copy images and write YOLO label/seg files."""
    skipped = 0
    for img_file in file_list:
        ann_file = img_file.replace(".jpg", ".json")
        ann_path = os.path.join(src_ann_dir, ann_file)
        img_path = os.path.join(src_img_dir, img_file)
        if not os.path.exists(ann_path) or not os.path.exists(img_path):
            skipped += 1
            continue

        with open(ann_path) as fh:
            data = json.load(fh)

        img = Image.open(img_path)
        W, H = img.size

        lines = []
        for v in data.values():
            if not isinstance(v, dict):
                continue
            cat = v.get("category_id")
            if cat not in top5:
                continue
            cls_idx = CAT_TO_IDX[cat]
            seg = v.get("segmentation", [])
            if not seg or len(seg[0]) < 6:
                continue
            pts = seg[0]
            poly_x = pts[0::2]
            poly_y = pts[1::2]
            cx, cy, bw, bh = poly_to_bbox_norm(poly_x, poly_y, W, H)
            seg_pts = polygon_to_yolo_seg(poly_x, poly_y, W, H)
            seg_str = " ".join(f"{p:.6f}" for p in seg_pts)
            lines.append(f"{cls_idx} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f} {seg_str}")

        if not lines:
            skipped += 1
            continue

        stem = Path(img_file).stem
        shutil.copy(img_path, os.path.join(dst_img_dir, img_file))
        with open(os.path.join(dst_lbl_dir, stem + ".txt"), "w") as fh:
            fh.write("\n".join(lines))

    print(f"  Skipped: {skipped}")

# directories
for d in [
    os.path.join(YOLO_DIR, "images/train"),
    os.path.join(YOLO_DIR, "images/val"),
    os.path.join(YOLO_DIR, "labels/train"),
    os.path.join(YOLO_DIR, "labels/val"),
]:
    os.makedirs(d, exist_ok=True)

print("Writing train set …")
write_split(train_files, SRC_TRAIN_IMG, SRC_TRAIN_ANN,
            os.path.join(YOLO_DIR, "images/train"),
            os.path.join(YOLO_DIR, "labels/train"))

print("Writing val set …")
write_split(val_files, SRC_TRAIN_IMG, SRC_TRAIN_ANN,
            os.path.join(YOLO_DIR, "images/val"),
            os.path.join(YOLO_DIR, "labels/val"))

# YAML config
data_yaml = {
    "path"  : YOLO_DIR,
    "train" : "images/train",
    "val"   : "images/val",
    "nc"    : NUM_CLASSES,
    "names" : list(IDX_TO_NAME.values()),
}
yaml_path = os.path.join(YOLO_DIR, "data.yaml")
with open(yaml_path, "w") as fh:
    yaml.dump(data_yaml, fh, default_flow_style=False)
print("data.yaml written →", yaml_path)

Writing train set …
  Skipped: 0
Writing val set …
  Skipped: 0
data.yaml written → /kaggle/working/yolo_dataset/data.yaml


In [5]:
# ── 4. Training Helper ───────────────────────────────────────
def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def train_yolo(model_weights, run_name, freeze_layers=0,
               epochs=NUM_EPOCHS, lr=LR_TRANSFER):
    """
    Train a YOLOv8-seg model.
    freeze_layers > 0  → transfer learning
    freeze_layers == 0 → from scratch
    """
    free_memory()
    model = YOLO(model_weights)

    results = model.train(
        data        = yaml_path,
        epochs      = epochs,
        imgsz       = IMG_SIZE,
        batch       = BATCH_SIZE,
        lr0         = lr,
        freeze      = freeze_layers if freeze_layers > 0 else None,
        name        = run_name,
        project     = OUTPUT_DIR,
        device      = DEVICE,
        workers     = 8,          # ← critical: no extra subprocesses
        cache       = False,      # ← don't cache dataset in RAM
        amp         = True, 
        close_mosaic  = 3,
        mixup         = 0.0,
        copy_paste    = 0.0,
        plots       = True,
        save        = True,
        exist_ok    = True,
        verbose     = False,
    )
    free_memory()
    return model, results

In [6]:
# ── 5a. Transfer Learning ────────────────────────────────────
print("\n" + "="*60)
print("TRANSFER LEARNING  (pretrained YOLOv8n-seg, freeze=10)")
print("="*60)
model_tl, results_tl = train_yolo(
    model_weights = "yolov8n-seg.pt",   # downloads ~6 MB
    run_name      = "yolo_transfer",
    freeze_layers = 10,
    epochs        = NUM_EPOCHS,
    lr            = LR_TRANSFER,
)


TRANSFER LEARNING  (pretrained YOLOv8n-seg, freeze=10)
Ultralytics 8.4.31 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=3, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_transfer, nbs=64, nms=False, opset=None, optimi

In [7]:
# ── 5b. From Scratch ─────────────────────────────────────────
print("\n" + "="*60)
print("FROM SCRATCH  (random weights, no freeze)")
print("="*60)
model_sc, results_sc = train_yolo(
    model_weights = "yolov8n-seg.yaml",  # architecture only, no weights
    run_name      = "yolo_scratch",
    freeze_layers = 0,
    epochs        = NUM_EPOCHS,
    lr            = LR_SCRATCH,
)


FROM SCRATCH  (random weights, no freeze)
Ultralytics 8.4.31 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=3, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_scratch, nbs=64, nms=False, opset=None, optimize=False, o

In [13]:

# ── 6. Evaluation Utilities ──────────────────────────────────

def get_val_predictions(model, val_img_dir, val_lbl_dir):
    """
    Run model on val images and collect per-image GT + pred lists.
    Returns:
      all_gt_cls  : list of GT class indices per image
      all_pr_cls  : list of predicted class indices (highest-conf box) per image
      all_gt_boxes: list of [N,4] arrays  (x1y1x2y2, pixel)
      all_pr_boxes: list of [M,4] arrays
      all_pr_confs: list of [M] arrays
      all_pr_cls_raw: list of [M] arrays (per-box class)
      img_sizes   : list of (W,H)
    """
    img_files = [f for f in os.listdir(val_img_dir) if f.endswith(".jpg")]
    all_gt_cls, all_pr_cls = [], []
    all_gt_boxes, all_pr_boxes, all_pr_confs, all_pr_cls_raw = [], [], [], []
    img_sizes = []

    for img_file in img_files:
        img_path = os.path.join(val_img_dir, img_file)
        lbl_path = os.path.join(val_lbl_dir, img_file.replace(".jpg", ".txt"))

        img = Image.open(img_path)
        W, H = img.size
        img_sizes.append((W, H))

        # --- ground truth ---
        gt_cls_list, gt_boxes = [], []
        if os.path.exists(lbl_path):
            with open(lbl_path) as fh:
                for line in fh:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    c = int(parts[0])
                    cx, cy, bw, bh = map(float, parts[1:5])
                    x1 = (cx - bw / 2) * W
                    y1 = (cy - bh / 2) * H
                    x2 = (cx + bw / 2) * W
                    y2 = (cy + bh / 2) * H
                    gt_cls_list.append(c)
                    gt_boxes.append([x1, y1, x2, y2])

        all_gt_cls.append(gt_cls_list)
        all_gt_boxes.append(np.array(gt_boxes) if gt_boxes else np.zeros((0, 4)))

        # --- predictions ---
        res = model.predict(img_path, conf=CONF_THRESHOLD,
                            iou=IOU_THRESHOLD, verbose=False, half=(DEVICE=="cuda"))
        boxes = res[0].boxes
        if boxes is not None and len(boxes) > 0:
            pr_boxes_arr = boxes.xyxy.cpu().numpy()
            pr_conf_arr  = boxes.conf.cpu().numpy()
            pr_cls_arr   = boxes.cls.cpu().numpy().astype(int)
        else:
            pr_boxes_arr = np.zeros((0, 4))
            pr_conf_arr  = np.array([])
            pr_cls_arr   = np.array([], dtype=int)

        all_pr_boxes.append(pr_boxes_arr)
        all_pr_confs.append(pr_conf_arr)
        all_pr_cls_raw.append(pr_cls_arr)

        # image-level class: highest-conf prediction
        pred_cls = int(pr_cls_arr[np.argmax(pr_conf_arr)]) if len(pr_conf_arr) > 0 else -1
        all_pr_cls.append(pred_cls)

    return (all_gt_cls, all_pr_cls, all_gt_boxes,
            all_pr_boxes, all_pr_confs, all_pr_cls_raw, img_sizes)


def iou_box(b1, b2):
    """IoU between two boxes [x1,y1,x2,y2]."""
    ix1 = max(b1[0], b2[0]); iy1 = max(b1[1], b2[1])
    ix2 = min(b1[2], b2[2]); iy2 = min(b1[3], b2[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    a1 = (b1[2]-b1[0]) * (b1[3]-b1[1])
    a2 = (b2[2]-b2[0]) * (b2[3]-b2[1])
    return inter / (a1 + a2 - inter + 1e-6)


def compute_detection_metrics(all_gt_cls, all_gt_boxes,
                               all_pr_cls_raw, all_pr_boxes,
                               all_pr_confs, iou_thresh=0.5):
    """
    Compute per-class AP (PASCAL), macro-mAP, per-class F1.
    Returns dict of metrics.
    """
    class_aps, class_f1s = {}, {}
    all_tp_fp = defaultdict(list)  # cls → list of (conf, is_tp)

    for gt_cls_list, gt_boxes, pr_cls, pr_boxes, pr_confs in zip(
            all_gt_cls, all_gt_boxes,
            all_pr_cls_raw, all_pr_boxes, all_pr_confs):

        gt_matched = set()
        for i, (pc, pb, pf) in enumerate(
                sorted(zip(pr_cls, pr_boxes, pr_confs),
                       key=lambda x: -x[2])):
            matched = False
            for j, (gc, gb) in enumerate(zip(gt_cls_list, gt_boxes)):
                if j in gt_matched or gc != pc:
                    continue
                if iou_box(pb, gb) >= iou_thresh:
                    gt_matched.add(j)
                    matched = True
                    break
            all_tp_fp[pc].append((pf, 1 if matched else 0))

    for c in range(NUM_CLASSES):
        entries = sorted(all_tp_fp[c], key=lambda x: -x[0])
        if not entries:
            class_aps[c]  = 0.0
            class_f1s[c]  = 0.0
            continue
        confs = np.array([e[0] for e in entries])
        tps   = np.array([e[1] for e in entries])
        cum_tp = np.cumsum(tps)
        cum_fp = np.cumsum(1 - tps)
        n_gt   = sum(sum(1 for g in gl if g == c)
                     for gl in all_gt_cls)
        prec = cum_tp / (cum_tp + cum_fp + 1e-6)
        rec  = cum_tp / (n_gt + 1e-6)
        # 11-point AP
        ap = 0.0
        for t in np.linspace(0, 1, 11):
            p = prec[rec >= t].max() if (rec >= t).any() else 0.0
            ap += p / 11
        class_aps[c] = ap
        # F1 at 0.5 conf
        half_mask = confs >= 0.5
        if half_mask.sum() > 0:
            tp_h = tps[half_mask].sum()
            fp_h = (1 - tps[half_mask]).sum()
            fn_h = n_gt - tp_h
            p_h  = tp_h / (tp_h + fp_h + 1e-6)
            r_h  = tp_h / (tp_h + fn_h + 1e-6)
            class_f1s[c] = 2 * p_h * r_h / (p_h + r_h + 1e-6)
        else:
            class_f1s[c] = 0.0

    mAP = np.mean(list(class_aps.values()))
    return {"class_ap": class_aps, "mAP": mAP, "class_f1": class_f1s}


def compute_seg_metrics(model, val_img_dir, val_lbl_dir):
    """
    Approximates per-class mIoU and Dice from predicted masks vs
    bounding-box proxy masks (since we only have polygon GT).
    For a full mIoU you'd rasterise the GT polygon; here we use
    the GT box as a proxy for IoU computation.
    """
    from PIL import ImageDraw
    iou_per_class  = defaultdict(list)
    dice_per_class = defaultdict(list)

    for img_file in os.listdir(val_img_dir):
        if not img_file.endswith(".jpg"):
            continue
        lbl_path = os.path.join(val_lbl_dir, img_file.replace(".jpg", ".txt"))
        img_path = os.path.join(val_img_dir, img_file)

        img = Image.open(img_path)
        W, H = img.size

        # GT masks (polygon rasterised)
        gt_masks = defaultdict(lambda: np.zeros((H, W), dtype=np.uint8))
        if os.path.exists(lbl_path):
            with open(lbl_path) as fh:
                for line in fh:
                    parts = line.strip().split()
                    if len(parts) < 7:
                        continue
                    c = int(parts[0])
                    coords = list(map(float, parts[5:]))
                    poly = [(coords[i]*W, coords[i+1]*H)
                            for i in range(0, len(coords)-1, 2)]
                    mask_img = Image.new("L", (W, H), 0)
                    ImageDraw.Draw(mask_img).polygon(poly, fill=1)
                    gt_masks[c] = np.maximum(gt_masks[c],
                                             np.array(mask_img, dtype=np.uint8))

        # Predicted masks
        res = model.predict(img_path, conf=CONF_THRESHOLD,
                            iou=IOU_THRESHOLD, verbose=False,
                            half=(DEVICE=="cuda"))
        pred_masks_raw = res[0].masks
        pred_boxes     = res[0].boxes

        pr_masks = defaultdict(lambda: np.zeros((H, W), dtype=np.uint8))
        if pred_masks_raw is not None and pred_boxes is not None:
            for mask_tensor, cls_tensor in zip(
                    pred_masks_raw.data, pred_boxes.cls):
                c   = int(cls_tensor.item())
                m   = mask_tensor.cpu().numpy()
                m_r = np.array(Image.fromarray((m * 255).astype(np.uint8))
                               .resize((W, H), Image.NEAREST)) > 127
                pr_masks[c] = np.maximum(pr_masks[c], m_r.astype(np.uint8))

        for c in range(NUM_CLASSES):
            gt_m = gt_masks[c]
            pr_m = pr_masks[c]
            inter = (gt_m & pr_m).sum()
            union = (gt_m | pr_m).sum()
            iou_per_class[c].append(inter / (union + 1e-6))
            dice_per_class[c].append(2*inter / (gt_m.sum() + pr_m.sum() + 1e-6))

    results = {}
    mious, dices = [], []
    for c in range(NUM_CLASSES):
        miou = np.mean(iou_per_class[c])
        dice = np.mean(dice_per_class[c])
        results[IDX_TO_NAME[c]] = {"mIoU": miou, "Dice": dice}
        mious.append(miou); dices.append(dice)

    results["macro_mIoU"] = np.mean(mious)
    results["macro_Dice"] = np.mean(dices)
    return results

In [14]:
# ── 7. Run Evaluation ────────────────────────────────────────
val_img_dir = os.path.join(YOLO_DIR, "images/val")
val_lbl_dir = os.path.join(YOLO_DIR, "labels/val")

def full_eval(model, tag):
    print(f"\n{'='*60}")
    print(f"Evaluating: {tag}")
    print('='*60)

    (all_gt_cls, all_pr_cls,
     all_gt_boxes, all_pr_boxes,
     all_pr_confs, all_pr_cls_raw, _) = get_val_predictions(
        model, val_img_dir, val_lbl_dir)

    # ── Detection metrics ──────────────────────────────────
    det = compute_detection_metrics(
        all_gt_cls, all_gt_boxes,
        all_pr_cls_raw, all_pr_boxes, all_pr_confs)

    print(f"\n[Detection] mAP@0.5 = {det['mAP']:.4f}")
    det_rows = []
    for c in range(NUM_CLASSES):
        name = IDX_TO_NAME[c]
        print(f"  {name:22s}  AP={det['class_ap'][c]:.4f}  F1={det['class_f1'][c]:.4f}")
        det_rows.append({"class": name,
                         "AP@0.5": round(det['class_ap'][c], 4),
                         "F1":     round(det['class_f1'][c], 4)})
    det_df = pd.DataFrame(det_rows)
    det_df.to_csv(os.path.join(RESULTS_DIR, f"{tag}_detection.csv"), index=False)

    # ── Segmentation metrics ───────────────────────────────
    seg = compute_seg_metrics(model, val_img_dir, val_lbl_dir)
    print(f"\n[Segmentation] macro mIoU={seg['macro_mIoU']:.4f}  "
          f"macro Dice={seg['macro_Dice']:.4f}")
    seg_rows = []
    for c in range(NUM_CLASSES):
        name = IDX_TO_NAME[c]
        print(f"  {name:22s}  mIoU={seg[name]['mIoU']:.4f}  "
              f"Dice={seg[name]['Dice']:.4f}")
        seg_rows.append({"class": name,
                         "mIoU": round(seg[name]['mIoU'], 4),
                         "Dice": round(seg[name]['Dice'], 4)})
    seg_rows.append({"class": "MACRO",
                     "mIoU": round(seg['macro_mIoU'], 4),
                     "Dice": round(seg['macro_Dice'], 4)})
    seg_df = pd.DataFrame(seg_rows)
    seg_df.to_csv(os.path.join(RESULTS_DIR, f"{tag}_segmentation.csv"), index=False)

    # ── ROC / AUC (image-level, one-vs-rest) ──────────────
    flat_gt  = []
    flat_pr  = []
    flat_conf = []
    for gt_list, pr_list, conf_list in zip(
            all_gt_cls, all_pr_cls_raw, all_pr_confs):
        for c in range(NUM_CLASSES):
            gt_bin = 1 if c in gt_list else 0
            if len(conf_list) > 0:
                class_conf = max(
                    (conf_list[i] for i, pc in enumerate(pr_list) if pc == c),
                    default=0.0)
            else:
                class_conf = 0.0
            flat_gt.append((c, gt_bin))
            flat_conf.append((c, class_conf))

    fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(4*NUM_CLASSES, 4))
    auc_scores = {}
    for c in range(NUM_CLASSES):
        y_true = [gt for cls, gt in flat_gt if cls == c]
        y_score= [sc for cls, sc in flat_conf if cls == c]
        if len(set(y_true)) < 2:
            auc_scores[IDX_TO_NAME[c]] = float('nan')
            continue
        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc = auc(fpr, tpr)
        auc_scores[IDX_TO_NAME[c]] = round(roc_auc, 4)
        axes[c].plot(fpr, tpr, label=f"AUC={roc_auc:.2f}")
        axes[c].plot([0,1],[0,1],"k--")
        axes[c].set_title(IDX_TO_NAME[c])
        axes[c].set_xlabel("FPR"); axes[c].set_ylabel("TPR")
        axes[c].legend()

    plt.suptitle(f"ROC Curves — {tag}", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"{tag}_roc.png"), dpi=100)
    plt.close()
    print(f"\n[AUC per class]: {auc_scores}")

    # ── Summary bar chart ──────────────────────────────────
    names = list(IDX_TO_NAME.values())
    ap_vals   = [det['class_ap'][c] for c in range(NUM_CLASSES)]
    miou_vals = [seg[IDX_TO_NAME[c]]['mIoU'] for c in range(NUM_CLASSES)]
    dice_vals = [seg[IDX_TO_NAME[c]]['Dice'] for c in range(NUM_CLASSES)]

    x = np.arange(NUM_CLASSES)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - 0.25, ap_vals,   0.25, label="AP@0.5")
    ax.bar(x,        miou_vals, 0.25, label="mIoU")
    ax.bar(x + 0.25, dice_vals, 0.25, label="Dice")
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=20, ha="right")
    ax.set_ylim(0, 1)
    ax.set_title(f"Per-Class Metrics — {tag}")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"{tag}_metrics_bar.png"), dpi=100)
    plt.close()

    free_memory()
    return {"detection": det, "segmentation": seg, "auc": auc_scores}


results_tl_eval = full_eval(model_tl, "transfer_learning")
results_sc_eval = full_eval(model_sc, "from_scratch")


Evaluating: transfer_learning

[Detection] mAP@0.5 = 0.4440
  short_sleeve_top        AP=0.6690  F1=0.7210
  trousers                AP=0.4580  F1=0.5143
  shorts                  AP=0.2028  F1=0.2884
  long_sleeve_top         AP=0.5416  F1=0.5635
  skirt                   AP=0.3485  F1=0.4359

[Segmentation] macro mIoU=0.1862  macro Dice=0.2141
  short_sleeve_top        mIoU=0.3224  Dice=0.3629
  trousers                mIoU=0.2191  Dice=0.2623
  shorts                  mIoU=0.1332  Dice=0.1589
  long_sleeve_top         mIoU=0.1317  Dice=0.1504
  skirt                   mIoU=0.1246  Dice=0.1361

[AUC per class]: {'short_sleeve_top': np.float64(0.8638), 'trousers': np.float64(0.935), 'shorts': np.float64(0.8936), 'long_sleeve_top': np.float64(0.8117), 'skirt': np.float64(0.8221)}

Evaluating: from_scratch

[Detection] mAP@0.5 = 0.3098
  short_sleeve_top        AP=0.5441  F1=0.5131
  trousers                AP=0.3590  F1=0.3879
  shorts                  AP=0.1365  F1=0.2112
  long_slee

In [15]:
# ── 8. Comparison Table ──────────────────────────────────────
print("\n" + "="*60)
print("FINAL COMPARISON SUMMARY")
print("="*60)

rows = []
for c in range(NUM_CLASSES):
    name = IDX_TO_NAME[c]
    rows.append({
        "Class"         : name,
        "TL_AP@0.5"     : round(results_tl_eval["detection"]["class_ap"][c], 4),
        "SC_AP@0.5"     : round(results_sc_eval["detection"]["class_ap"][c], 4),
        "TL_F1_det"     : round(results_tl_eval["detection"]["class_f1"][c], 4),
        "SC_F1_det"     : round(results_sc_eval["detection"]["class_f1"][c], 4),
        "TL_mIoU"       : round(results_tl_eval["segmentation"][name]["mIoU"], 4),
        "SC_mIoU"       : round(results_sc_eval["segmentation"][name]["mIoU"], 4),
        "TL_Dice"       : round(results_tl_eval["segmentation"][name]["Dice"], 4),
        "SC_Dice"       : round(results_sc_eval["segmentation"][name]["Dice"], 4),
    })

# macro row
rows.append({
    "Class"     : "MACRO",
    "TL_AP@0.5" : round(results_tl_eval["detection"]["mAP"], 4),
    "SC_AP@0.5" : round(results_sc_eval["detection"]["mAP"], 4),
    "TL_F1_det" : round(np.mean(list(results_tl_eval["detection"]["class_f1"].values())), 4),
    "SC_F1_det" : round(np.mean(list(results_sc_eval["detection"]["class_f1"].values())), 4),
    "TL_mIoU"   : round(results_tl_eval["segmentation"]["macro_mIoU"], 4),
    "SC_mIoU"   : round(results_sc_eval["segmentation"]["macro_mIoU"], 4),
    "TL_Dice"   : round(results_tl_eval["segmentation"]["macro_Dice"], 4),
    "SC_Dice"   : round(results_sc_eval["segmentation"]["macro_Dice"], 4),
})

comparison_df = pd.DataFrame(rows)
print(comparison_df.to_string(index=False))
comparison_df.to_csv(os.path.join(RESULTS_DIR, "comparison_summary.csv"), index=False)
print(f"\nAll results saved to: {RESULTS_DIR}")




FINAL COMPARISON SUMMARY
           Class  TL_AP@0.5  SC_AP@0.5  TL_F1_det  SC_F1_det  TL_mIoU  SC_mIoU  TL_Dice  SC_Dice
short_sleeve_top     0.6690     0.5441     0.7210     0.5131   0.3224   0.2463   0.3629   0.2785
        trousers     0.4580     0.3590     0.5143     0.3879   0.2191   0.1760   0.2623   0.2134
          shorts     0.2028     0.1365     0.2884     0.2112   0.1332   0.1178   0.1589   0.1379
 long_sleeve_top     0.5416     0.2607     0.5635     0.1584   0.1317   0.0750   0.1504   0.0859
           skirt     0.3485     0.2490     0.4359     0.2631   0.1246   0.0803   0.1361   0.0873
           MACRO     0.4440     0.3098     0.5046     0.3068   0.1862   0.1391   0.2141   0.1606

All results saved to: /kaggle/working/results


In [16]:
# ── 9. Qualitative Visualisation (4 sample images) ───────────
print("\nGenerating qualitative visualisation …")
sample_imgs = random.sample(
    [f for f in os.listdir(val_img_dir) if f.endswith(".jpg")],
    min(4, len(os.listdir(val_img_dir))))

fig, axes = plt.subplots(len(sample_imgs), 3,
                         figsize=(15, 5 * len(sample_imgs)))
COLORS = plt.cm.get_cmap("tab10", NUM_CLASSES)

for row_i, img_file in enumerate(sample_imgs):
    img_path = os.path.join(val_img_dir, img_file)
    img_np   = np.array(Image.open(img_path).convert("RGB"))

    ax_gt  = axes[row_i][0] if len(sample_imgs) > 1 else axes[0]
    ax_tl  = axes[row_i][1] if len(sample_imgs) > 1 else axes[1]
    ax_sc  = axes[row_i][2] if len(sample_imgs) > 1 else axes[2]

    # GT
    lbl_path = os.path.join(val_lbl_dir, img_file.replace(".jpg", ".txt"))
    ax_gt.imshow(img_np)
    if os.path.exists(lbl_path):
        H, W = img_np.shape[:2]
        with open(lbl_path) as fh:
            for line in fh:
                parts = line.strip().split()
                if len(parts) < 5: continue
                c = int(parts[0])
                cx,cy,bw,bh = map(float, parts[1:5])
                x1=(cx-bw/2)*W; y1=(cy-bh/2)*H
                rect = mpatches.Rectangle(
                    (x1,y1), bw*W, bh*H,
                    linewidth=2, edgecolor=COLORS(c),
                    facecolor="none")
                ax_gt.add_patch(rect)
                ax_gt.text(x1, y1-2, IDX_TO_NAME[c],
                           color=COLORS(c), fontsize=7, weight="bold")
    ax_gt.set_title("Ground Truth", fontsize=9)
    ax_gt.axis("off")

    for ax_m, m, label in [
            (ax_tl, model_tl, "Transfer Learning"),
            (ax_sc, model_sc, "From Scratch")]:
        res = m.predict(img_path, conf=CONF_THRESHOLD,
                        iou=IOU_THRESHOLD, verbose=False,
                        half=(DEVICE=="cuda"))
        overlay = img_np.copy()
        if res[0].masks is not None:
            for mask_t, cls_t in zip(res[0].masks.data, res[0].boxes.cls):
                c   = int(cls_t.item())
                msk = np.array(Image.fromarray(
                    (mask_t.cpu().numpy()*255).astype(np.uint8))
                    .resize((img_np.shape[1], img_np.shape[0]),
                            Image.NEAREST)) > 127
                colour = (np.array(COLORS(c)[:3]) * 255).astype(np.uint8)
                overlay[msk] = (0.5*overlay[msk] + 0.5*colour).astype(np.uint8)
        ax_m.imshow(overlay)
        if res[0].boxes is not None:
            for box, cls_t, conf_t in zip(
                    res[0].boxes.xyxy.cpu().numpy(),
                    res[0].boxes.cls.cpu().numpy(),
                    res[0].boxes.conf.cpu().numpy()):
                c = int(cls_t)
                rect = mpatches.Rectangle(
                    (box[0], box[1]), box[2]-box[0], box[3]-box[1],
                    linewidth=2, edgecolor=COLORS(c), facecolor="none")
                ax_m.add_patch(rect)
                ax_m.text(box[0], box[1]-2,
                          f"{IDX_TO_NAME[c]} {conf_t:.2f}",
                          color=COLORS(c), fontsize=7, weight="bold")
        ax_m.set_title(label, fontsize=9)
        ax_m.axis("off")

plt.suptitle("Qualitative Results: GT | Transfer Learning | From Scratch",
             fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "qualitative_results.png"),
            dpi=100, bbox_inches="tight")
plt.close()
print("Saved qualitative_results.png")
print("\n✅  All done! Check:", RESULTS_DIR)


Generating qualitative visualisation …


/tmp/ipykernel_55/3807299491.py:9: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  COLORS = plt.cm.get_cmap("tab10", NUM_CLASSES)


Saved qualitative_results.png

✅  All done! Check: /kaggle/working/results
